##ASSIGNMENT-2

Ques-1 Identify !, %, and %% used in cell in Google Colab.

! for terminal access

In [1]:
! ls

sample_data


In [2]:
! pwd

/content


% for single line

In [3]:
%pwd

'/content'

In [4]:
%matplotlib inline

%% for entire cell

In [5]:
%%writefile test.py
print("Saved to file")

Writing test.py


Q-2.  Identify all key nvidia-smi commands with multiple options

In [6]:
!nvidia-smi


Sun Feb 15 09:49:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-20cbf7b3-48c8-65ae-75b0-0fe9b182dac9)


In [14]:
!nvidia-smi -l

Sun Feb 15 09:54:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
!nvidia-smi pmon

# gpu         pid   type     sm    mem    enc    dec    jpg    ofa    command 
# Idx           #    C/G      %      %      %      %      %      %    name 
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              


Ques-3 Debug common CUDA errors (zero output, incorrect indexing, PTX errors)

Zero output error

In [16]:
%%writefile hello_error.cu
#include <stdio.h>

__global__ void mykernel(void)
{
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main(void)
{
    mykernel<<<1,1>>>();
    printf("Hello World with cpu\n");
    return 0;
}


Writing hello_error.cu


In [17]:
!nvcc -arch=sm_75 hello_error.cu -o hello
! ./hello

Hello World with cpu


Fixed Code:

In [18]:
%%writefile hello_fixed.cu
#include <stdio.h>

__global__ void mykernel(void)
{
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main(void)
{
    mykernel<<<1,1>>>();
    cudaDeviceSynchronize();
    printf("Hello World with cpu\n");
    return 0;
}


Writing hello_fixed.cu


In [19]:
!nvcc -arch=sm_75 hello_fixed.cu -o fixed
! ./fixed

Hello from GPU thread 0
Hello World with cpu


Incorrect Indexing

In [20]:
%%writefile wrongindex.cu
#include<stdio.h>
__global__ void mykernel(void){
int id=threadIdx.x;
printf("Thread %d\n", id);
};
int main(void)
{
mykernel<<<3,4>>>();
cudaDeviceSynchronize();
return 0;
}

Writing wrongindex.cu


In [21]:
!nvcc -arch=sm_75 wrongindex.cu -o wrong
! ./wrong

Thread 0
Thread 1
Thread 2
Thread 3
Thread 0
Thread 1
Thread 2
Thread 3
Thread 0
Thread 1
Thread 2
Thread 3


Fixed Indexing:

In [22]:
%%writefile fixedindex.cu
#include<stdio.h>
__global__ void mykernel(void){
int id=blockIdx.x*blockDim.x+threadIdx.x;
printf("Thread %d\n", id);
};
int main(void)
{
mykernel<<<3,4>>>();
cudaDeviceSynchronize();
return 0;
}

Writing fixedindex.cu


In [23]:
!nvcc -arch=sm_75 fixedindex.cu -o fix
! ./fix

Thread 4
Thread 5
Thread 6
Thread 7
Thread 0
Thread 1
Thread 2
Thread 3
Thread 8
Thread 9
Thread 10
Thread 11


PTX error

In [24]:
%%writefile ptx_error.cu
#include<stdio.h>

__global__ void kernel(){}

int main(){
    kernel<<<1,2000>>>();
    cudaDeviceSynchronize();
    printf("%s\n", cudaGetErrorString(cudaGetLastError()));
    return 0;
}


Writing ptx_error.cu


In [25]:
!nvcc -arch=sm_75 ptx_error.cu -o ptx
! ./ptx

invalid configuration argument


Fixed:


In [26]:
%%writefile ptx_fixed.cu
#include<stdio.h>

__global__ void k(){}

int main(){
    k<<<1,256>>>();
    cudaDeviceSynchronize();
    printf("%s\n", cudaGetErrorString(cudaGetLastError()));
    return 0;
}


Writing ptx_fixed.cu


In [27]:
!nvcc -arch=sm_75 ptx_fixed.cu -o ptxfixed
! ./ptxfixed

no error


Ques-4 Write a CUDA C/C++ program to demonstrate GPU kernel execu on and thread indexing.

a. Launch a CUDA kernel using: 1 block and 8 threads

b. Each thread must print: Hello from GPU thread <global_thread_id>

c. Compute the global thread ID using: global_thread_id = blockIdx.x * blockDim.x + threadIdx.x

d. Clearly separate: Host code (CPU) & Device code (GPU kernel)

In [28]:
%%writefile globalthread.cu
#include<stdio.h>
__global__ void mykernel()
{
    int global_thread_id=blockIdx.x*blockDim.x+threadIdx.x;
    printf("Hello from GPU thread %d\n", global_thread_id);
}

int main(void)
{
    mykernel<<<1,8>>>();
    cudaDeviceSynchronize();
    return 0;
}


Writing globalthread.cu


In [29]:
!nvcc -arch=sm_75 globalthread.cu -o global
! ./global

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3
Hello from GPU thread 4
Hello from GPU thread 5
Hello from GPU thread 6
Hello from GPU thread 7


Ques-5 Write a CUDA program to demonstrate host and device memory separation.

a. Create an integer array of size 5 on the host (CPU).

b. Allocate corresponding memory on the device (GPU) using cudaMalloc().

c. Copy data from host to device using cudaMemcpy().

d. Launch a kernel where GPU threads print values from device memory.

e. Copy the data back from device to host and print it on CPU.

In [30]:
%%writefile host_device.cu
#include<stdio.h>

__global__ void printGPU(int *d_arr)
{
    int i=threadIdx.x;
    printf("GPU thread %d value = %d\n", i, d_arr[i]);
}


int main()
{
    const int N=5;
    int h_arr[N]={10,20,30,40,50};
    int *d_arr;

    cudaMalloc(&d_arr, N*sizeof(int));
    cudaMemcpy(d_arr, h_arr, N*sizeof(int), cudaMemcpyHostToDevice);

    printGPU<<<1,N>>>(d_arr);
    cudaDeviceSynchronize();

    cudaMemcpy(h_arr, d_arr, N*sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nCPU printing values:\n");
    for(int i=0;i<N;i++)
        printf("%d ", h_arr[i]);
    cudaFree(d_arr);
    return 0;
}


Writing host_device.cu


In [31]:
!nvcc -arch=sm_75 host_device.cu -o array
! ./array

GPU thread 0 value = 10
GPU thread 1 value = 20
GPU thread 2 value = 30
GPU thread 3 value = 40
GPU thread 4 value = 50

CPU printing values:
10 20 30 40 50 

Ques-6 Compare CPU mes of List/tuple with Numpy arrays.

In [32]:
import numpy as np

In [33]:
list1=[1,2,2,6,8,3,9,0,8,7,6,5,5]
list2=[5,6,7,3,4,3,3,6,7,3,2,5,9]

In [34]:
list_res=[]

In [35]:
%%timeit
for i in range(len(list1)):
  list_res.append(list1[i]*list2[i])

724 ns ± 12.2 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


In [36]:
arr1=np.array(list1)
arr2=np.array(list2)

In [38]:
%%timeit
arr3=arr1*arr2

618 ns ± 9.09 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)
